# Issue 11 Exploratory Data Analaysis (EDA) Notebook:

Goals:
- Recreate the SAS Mortality group score in Python
- Note: Before Program 1 runs, Program 0 has already done the preprocessing

Steps:
1. Load Program 0 data
2. Pick Mortality columns
3. Count available measures
4. Give them equal weight
5. Average them
6. Standardize the average
7. Compare Python to SAS (with a small enough tolerance to avoid mis-detection of semantic behavior differences)
8. Check missing-value behavior

In [1]:
# import needed libraries
import numpy as np
import pandas as pd

# load CSV file
df = pd.read_csv("../data/Project_1/SAS Output CSV/STD_DATA_2025JUL_ANALYSIS.csv")

In [ ]:
# via program-1-analysis.md: "Mortality" has 7 measures
# choose only "Mortality" domain measures

mortality_cols = [
    "std_MORT_30_AMI",
    "std_MORT_30_CABG",
    "std_MORT_30_COPD",
    "std_MORT_30_HF",
    "std_MORT_30_PN",
    "std_MORT_30_STK",
    "std_PSI_4_SURG_COMP",
]

In [3]:
# count how many Mortality measures each hospital has (not missing) across each row
total_cnt = df[mortality_cols].notna().sum(axis=1)

# calculate measure weight
measure_wt = 1 / total_cnt
measure_wt = measure_wt.where(total_cnt > 0)

# average Mortality measures for each hospital (missing values ignored)
# score_before_std is the group score before the final standardization step
score_before_std = df[mortality_cols].mean(axis=1)

In [4]:
# calculate sample mean and standard deviation
mean = score_before_std.mean()
standard_dev = score_before_std.std(ddof=1)

# standardize the score
grp_score = (score_before_std - mean) / standard_dev

print("Sample mean:", mean)
print("Sample standard Deviation:", standard_dev)

Sample mean: -0.07348350342882984
Sample standard Deviation: 0.7302551705532135


In [5]:
# view results in a table
result = pd.DataFrame(
    {
        "PROVIDER_ID": df["PROVIDER_ID"],
        "total_cnt": total_cnt,
        "measure_wt": measure_wt,
        "score_before_std": score_before_std,
        "Mean": mean,
        "StdDev": standard_dev,
        "grp_score": grp_score,
    }
)

result.head()

,PROVIDER_ID,total_cnt,measure_wt,score_before_std,Mean,StdDev,grp_score
0,010001,7,0.142857,0.142753,-0.073484,0.730255,0.296110
1,010005,6,0.166667,-0.689113,-0.073484,0.730255,-0.843034
2,010006,7,0.142857,-1.198571,-0.073484,0.730255,-1.540677
3,010007,3,0.333333,-2.505745,-0.073484,0.730255,-3.330700
4,010008,1,1.000000,-0.419012,-0.073484,0.730255,-0.473161


In [6]:
# compare SAS and Python
# result: what our Python code produced
# sas_trusted_file: reference

sas = pd.read_csv("../data/Project_1/SAS Output CSV/OUTCOME_MORTALITY.csv")

# match each hospital in Python with the same hospital in SAS
# keep only PROVIDER_ID and grp_score from the SAS file
compare = result.merge(
    sas[["PROVIDER_ID", "grp_score"]],
    on="PROVIDER_ID",
    suffixes=("_python", "_sas"),
)

# check whether the Python grp_score is close to the SAS grp_score
# atol=1e-6: allows tiny decimal differences
# equal_nan=True: treats missing values in both files as matching
same_scores = np.isclose(
    compare["grp_score_python"],
    compare["grp_score_sas"],
    atol=1e-6,
    equal_nan=True,
)

# print how many hospitals were compared
print("Total hospitals:", len(compare))

# print how many Python scores matched the SAS scores
print("Different scores:", (~same_scores).sum())

# print how many scores were different
print("All scores match:", same_scores.all())

Total hospitals: 4566
Different scores: 0
All scores match: True


In [7]:
# missing value check
zero_measures = result[result["total_cnt"] == 0]

print("Hospitals with 0 Mortality measures:", len(zero_measures))
print(zero_measures[["measure_wt", "score_before_std", "grp_score"]].head())

Hospitals with 0 Mortality measures: 884
    measure_wt  score_before_std  grp_score
8          NaN               NaN        NaN
11         NaN               NaN        NaN
16         NaN               NaN        NaN
25         NaN               NaN        NaN
44         NaN               NaN        NaN
